# 2016~2024 지방재정365 세출예산 원자료 구조 점검

작성자: 이정연
이슈 #69


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data/raw/지방재정365/기능별_회계별_세출예산"
EXPECTED_YEARS = list(range(2016, 2025))
KEY_COLUMNS = ["회계연도", "자치단체코드", "회계구분명", "분야코드"]
VALUE_COLUMNS = ["세출예산총계액", "세출예산순계액"]
EXPECTED_COLUMNS = [
    "No",
    "회계연도",
    "지역명",
    "자치단체코드",
    "자치단체명",
    "회계구분명",
    "분야코드",
    "분야명",
    *VALUE_COLUMNS,
]

paths = sorted(RAW_DIR.glob("*.csv"))
assert len(paths) == 9
print("input:", RAW_DIR.relative_to(ROOT), "/", len(paths), "files")

input: data/raw/지방재정365/기능별_회계별_세출예산 / 9 files


## Data

In [2]:
# 1. 연도별 파일 로드와 스키마 계약 검사
frames, file_profile = [], []
for path in paths:
    frame = pd.read_csv(path, encoding="utf-8-sig")
    file_year = int(path.name[:4])
    assert frame.columns.tolist() == EXPECTED_COLUMNS
    assert frame["회계연도"].nunique() == 1 and int(frame["회계연도"].iloc[0]) == file_year
    frame["출처파일"] = path.name
    frames.append(frame)
    file_profile.append(
        {
            "연도": file_year,
            "행": len(frame),
            "열": len(EXPECTED_COLUMNS),
            "파일크기_bytes": path.stat().st_size,
        }
    )

data = pd.concat(frames, ignore_index=True)
file_profile = pd.DataFrame(file_profile)
assert file_profile["연도"].tolist() == EXPECTED_YEARS
display(file_profile)
print("통합 행:", len(data))

,연도,행,열,파일크기_bytes
0,2016,5077,10,591163
1,2017,5037,10,587183
2,2018,4914,10,575165
3,2019,4958,10,579971
4,2020,4962,10,582133
5,2021,5011,10,587625
6,2022,4978,10,584606
7,2023,4968,10,583612
8,2024,4992,10,586316


통합 행: 44897


## Results

In [3]:
# 2. 결측·중복·값 유효성
quality_summary = pd.DataFrame(
    [
        {
            "행": len(data),
            "필수값결측": int(data[EXPECTED_COLUMNS].isna().any(axis=1).sum()),
            "복합키중복": int(data.duplicated(KEY_COLUMNS).sum()),
            "음수_총계액": int(data["세출예산총계액"].lt(0).sum()),
            "음수_순계액": int(data["세출예산순계액"].lt(0).sum()),
            "총계순계_모두0": int(data[VALUE_COLUMNS].eq(0).all(axis=1).sum()),
        }
    ]
)
assert (
    quality_summary[["필수값결측", "복합키중복", "음수_총계액", "음수_순계액"]].to_numpy().sum()
    == 0
)
display(quality_summary)

,행,필수값결측,복합키중복,음수_총계액,음수_순계액,총계순계_모두0
0,44897,0,0,0,0,1351


In [4]:
# 3. 연도별 지역·자치단체·회계·분야 완전성
coverage = (
    data.groupby("회계연도")
    .agg(
        행=("No", "size"),
        지역수=("지역명", "nunique"),
        자치단체수=("자치단체코드", "nunique"),
        회계구분수=("회계구분명", "nunique"),
        분야코드수=("분야코드", "nunique"),
    )
    .reset_index()
)
assert coverage["지역수"].eq(17).all()
assert coverage["자치단체수"].eq(243).all()
assert coverage["회계구분수"].eq(3).all()
assert coverage["분야코드수"].eq(14).all()
display(coverage)
print("회계구분:", sorted(data["회계구분명"].unique()))

,회계연도,행,지역수,자치단체수,회계구분수,분야코드수
0,2016,5077,17,243,3,14
1,2017,5037,17,243,3,14
2,2018,4914,17,243,3,14
3,2019,4958,17,243,3,14
4,2020,4962,17,243,3,14
5,2021,5011,17,243,3,14
6,2022,4978,17,243,3,14
7,2023,4968,17,243,3,14
8,2024,4992,17,243,3,14


회계구분: ['공기업특별회계', '기타특별회계', '일반회계']


In [5]:
# 4. 지역별 자치단체 수와 행정구역 변화
municipality_counts = (
    data.drop_duplicates(["회계연도", "지역명", "자치단체코드"])
    .groupby(["지역명", "회계연도"])
    .size()
    .unstack()
)
display(municipality_counts)

code_names = data[["회계연도", "지역명", "자치단체코드", "자치단체명"]].drop_duplicates()
name_changes = code_names.groupby("자치단체코드").filter(
    lambda group: group["자치단체명"].nunique() > 1
)
display(name_changes.sort_values(["자치단체코드", "회계연도"]))

회계연도,2016,2017,2018,2019,2020,2021,2022,2023,2024
지역명,,,,,,,,,
강원,19,19,19,19,19,19,19,19,19
경기,32,32,32,32,32,32,32,32,32
경남,19,19,19,19,19,19,19,19,19
경북,24,24,24,24,24,24,24,24,23
광주,6,6,6,6,6,6,6,6,6
대구,9,9,9,9,9,9,9,9,10
대전,6,6,6,6,6,6,6,6,6
부산,17,17,17,17,17,17,17,17,17
서울,26,26,26,26,26,26,26,26,26


,회계연도,지역명,자치단체코드,자치단체명
1036,2016,인천,2813000,인천남구
6108,2017,인천,2813000,인천남구
11109,2018,인천,2813000,인천미추홀구
16033,2019,인천,2813000,인천미추홀구
21006,2020,인천,2813000,인천미추홀구
25973,2021,인천,2813000,인천미추홀구
30984,2022,인천,2813000,인천미추홀구
35957,2023,인천,2813000,인천미추홀구
40950,2024,인천,2813000,인천미추홀구


In [6]:
# 5. 분야 코드 안정성과 명칭 변경
field_names = (
    data[["회계연도", "분야코드", "분야명"]].drop_duplicates().sort_values(["분야코드", "회계연도"])
)
field_name_changes = field_names.groupby("분야코드").filter(
    lambda group: group["분야명"].nunique() > 1
)
display(field_name_changes)

,회계연도,분야코드,분야명
1,2016,70,환경보호
5078,2017,70,환경보호
10114,2018,70,환경보호
15028,2019,70,환경보호
19986,2020,70,환경
24948,2021,70,환경
29959,2022,70,환경
34937,2023,70,환경
39905,2024,70,환경
11,2016,110,산업ㆍ중소기업


In [7]:
# 6. 회계구분 커버리지: 모든 자치단체에 세 회계가 반드시 존재하지는 않음
account_coverage = (
    data.groupby(["회계연도", "자치단체코드"])["회계구분명"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("보유회계수")
    .to_frame("자치단체연도수")
)
account_by_year = (
    data.groupby(["회계연도", "회계구분명"])
    .agg(행=("No", "size"), 자치단체수=("자치단체코드", "nunique"))
    .reset_index()
)
display(account_coverage)
display(account_by_year)

,자치단체연도수
보유회계수,
2,1055
3,1132


,회계연도,회계구분명,행,자치단체수
0,2016,공기업특별회계,353,129
1,2016,기타특별회계,1542,243
2,2016,일반회계,3182,243
3,2017,공기업특별회계,348,130
4,2017,기타특별회계,1511,243
5,2017,일반회계,3178,243
6,2018,공기업특별회계,316,123
7,2018,기타특별회계,1424,243
8,2018,일반회계,3174,243
9,2019,공기업특별회계,320,125


In [8]:
# 7. 총계·순계 관계는 집계 수준별로 다르게 평가
aggregation_qa = []
levels = {
    "원자료 분야행": ["회계연도", "자치단체코드", "회계구분명", "분야코드"],
    "자치단체×회계": ["회계연도", "자치단체코드", "회계구분명"],
    "자치단체": ["회계연도", "자치단체코드"],
    "지역": ["회계연도", "지역명"],
    "전국": ["회계연도"],
}
for level, keys in levels.items():
    aggregate = data.groupby(keys, as_index=False)[VALUE_COLUMNS].sum()
    aggregation_qa.append(
        {
            "집계수준": level,
            "행": len(aggregate),
            "총계<순계": int(aggregate["세출예산총계액"].lt(aggregate["세출예산순계액"]).sum()),
            "총계=순계": int(aggregate["세출예산총계액"].eq(aggregate["세출예산순계액"]).sum()),
        }
    )
aggregation_qa = pd.DataFrame(aggregation_qa)
display(aggregation_qa)

,집계수준,행,총계<순계,총계=순계
0,원자료 분야행,44897,2905,25263
1,자치단체×회계,5506,1559,517
2,자치단체,2187,1071,0
3,지역,153,0,0
4,전국,9,0,0


In [9]:
# 8. 153개 지역×연도 집계 후보
region_panel_candidate = data.groupby(["회계연도", "지역명"], as_index=False)[VALUE_COLUMNS].sum()
region_panel_candidate["총계_순계_차액"] = (
    region_panel_candidate["세출예산총계액"] - region_panel_candidate["세출예산순계액"]
)
region_panel_candidate["순계액_백만원"] = region_panel_candidate["세출예산순계액"] / 1_000_000
panel_qa = pd.DataFrame(
    [
        {
            "행": len(region_panel_candidate),
            "연도": region_panel_candidate["회계연도"].nunique(),
            "지역": region_panel_candidate["지역명"].nunique(),
            "키중복": int(region_panel_candidate.duplicated(["회계연도", "지역명"]).sum()),
            "결측": int(region_panel_candidate[VALUE_COLUMNS].isna().any(axis=1).sum()),
            "0이하_순계액": int(region_panel_candidate["세출예산순계액"].le(0).sum()),
            "총계<순계": int(region_panel_candidate["총계_순계_차액"].lt(0).sum()),
        }
    ]
)
assert panel_qa.iloc[0].to_dict() == {
    "행": 153,
    "연도": 9,
    "지역": 17,
    "키중복": 0,
    "결측": 0,
    "0이하_순계액": 0,
    "총계<순계": 0,
}
display(panel_qa)
display(region_panel_candidate.head(17))

,행,연도,지역,키중복,결측,0이하_순계액,총계<순계
0,153,9,17,0,0,0,0


,회계연도,지역명,세출예산총계액,세출예산순계액,총계_순계_차액,순계액_백만원
0,2016,강원,12439345023000,9415973114000,3023371909000,9.415973e+06
1,2016,경기,47124571518000,36249467525000,10875103993000,3.624947e+07
2,2016,경남,18897926368000,14026791169000,4871135199000,1.402679e+07
3,2016,경북,20737266442000,15225641095000,5511625347000,1.522564e+07
4,2016,광주,5917328537000,4106117093000,1811211444000,4.106117e+06
5,2016,대구,10142136000000,7213154474000,2928981526000,7.213154e+06
6,2016,대전,5636043167000,4016882713000,1619160454000,4.016883e+06
7,2016,부산,14696969244000,10573185713000,4123783531000,1.057319e+07
8,2016,서울,39210741290000,27534501217000,11676240073000,2.753450e+07
9,2016,세종,1117265816000,1048827518000,68438298000,1.048828e+06


In [10]:
# 9. 목적 적합성 판정표
fitness = pd.DataFrame(
    [
        {"검사항목": "9개년 동일 스키마", "판정": "PASS", "근거": "2016~2024 모두 동일 10컬럼"},
        {"검사항목": "17개 지역×9개년", "판정": "PASS", "근거": "153개 키 완전, 중복·결측 0"},
        {
            "검사항목": "총계·순계 동시 보존",
            "판정": "PASS",
            "근거": "두 금액 컬럼 모두 원 단위로 존재",
        },
        {
            "검사항목": "회계 범위",
            "판정": "PASS_WITH_CAVEAT",
            "근거": "일반·기타특별·공기업특별만 존재; 기금 없음",
        },
        {
            "검사항목": "지역 통합 순계 산식",
            "판정": "NEEDS_CROSSCHECK",
            "근거": "지역 내 자치단체·회계·분야 합산 후보는 생성 가능하나 공식 통합 순계와 교차검증 필요",
        },
        {
            "검사항목": "당초예산 여부",
            "판정": "BLOCKED",
            "근거": "CSV에 예산 단계 필드가 없어 파일만으로 당초·최종을 식별할 수 없음",
        },
    ]
)
display(fitness)

,검사항목,판정,근거
0,9개년 동일 스키마,PASS,2016~2024 모두 동일 10컬럼
1,17개 지역×9개년,PASS,"153개 키 완전, 중복·결측 0"
2,총계·순계 동시 보존,PASS,두 금액 컬럼 모두 원 단위로 존재
3,회계 범위,PASS_WITH_CAVEAT,일반·기타특별·공기업특별만 존재; 기금 없음
4,지역 통합 순계 산식,NEEDS_CROSSCHECK,지역 내 자치단체·회계·분야 합산 후보는 생성 가능하나 공식 통합 순계와 교차검증 필요
5,당초예산 여부,BLOCKED,CSV에 예산 단계 필드가 없어 파일만으로 당초·최종을 식별할 수 없음


## Takeaways

- 구조 QA는 통과했으며 153행 패널 생성이 가능하다.
- 행 단위 `총계≥순계` 규칙은 사용하지 않는다.
- 기금은 별도 자료가 필요하다.
- 상세 정의와 한계는 #69 구조점검 보고서를 따른다.
